In [60]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.framework.formats import landmark_pb2
from mediapipe import solutions
import numpy as np
import cv2
import pandas as pd
import math
import os

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
HandLandmarkerResult = mp.tasks.vision.HandLandmarkerResult
VisionRunningMode = mp.tasks.vision.RunningMode

In [2]:
def print_result(result: HandLandmarkerResult, output_image: mp.Image, timestamp_ms: int):
    print('hand landmarker result: {}'.format(result))

In [10]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO)
    # result_callback=print_result)

In [ ]:
video = cv2.VideoCapture("video_files/0.mp4")
dist_folder = "gesture_data/"

In [68]:
landmarks = [[lm.name+"_x", lm.name+"_y", lm.name+"_z"] for lm in mp.solutions.hands.HandLandmark]
landmarks_np = np.array(landmarks).flatten()
landmark_columns = landmarks_np.tolist()
landmark_columns.insert(0, 'frame_index')
landmark_columns.insert(1, 'hand_index')
print(landmark_columns)

for video_index, file in enumerate(os.listdir("video_files/")):
    full_path = os.path.join(os.getcwd(), "video_files", file)
    # print(full_path)
    video = cv2.VideoCapture(full_path)
    video_frame_rate = video.get(cv2.CAP_PROP_FPS)
    total_frames = video.get(cv2.CAP_PROP_FRAME_COUNT)

    with HandLandmarker.create_from_options(options) as landmarker:
        gesture_data = pd.DataFrame(columns=landmark_columns)
        
        for frame_index in range(0, int(total_frames)):
            video.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
            _, frame = video.read()
            
            if _:
                current_timestamp = int((frame_index / video_frame_rate) * 1000)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
                result = landmarker.detect_for_video(mp_image, current_timestamp)
                landmarks = result.hand_landmarks
                
                for landmark in landmarks:
                    nodes = []
                    nodes.append(frame_index)
                    nodes.append(result.handedness[0][0].index)
                    for pos in landmark:
                        try:
                            nodes.extend((pos.x, pos.y, pos.z))
                        
                        except:
                            print("no data")
                    
                    gesture_data = pd.concat([gesture_data, pd.DataFrame([nodes], columns=landmark_columns)])
        
        gesture_data.to_csv(f"gesture_data/{video_index}.csv")

['frame_index', 'hand_index', 'WRIST_x', 'WRIST_y', 'WRIST_z', 'THUMB_CMC_x', 'THUMB_CMC_y', 'THUMB_CMC_z', 'THUMB_MCP_x', 'THUMB_MCP_y', 'THUMB_MCP_z', 'THUMB_IP_x', 'THUMB_IP_y', 'THUMB_IP_z', 'THUMB_TIP_x', 'THUMB_TIP_y', 'THUMB_TIP_z', 'INDEX_FINGER_MCP_x', 'INDEX_FINGER_MCP_y', 'INDEX_FINGER_MCP_z', 'INDEX_FINGER_PIP_x', 'INDEX_FINGER_PIP_y', 'INDEX_FINGER_PIP_z', 'INDEX_FINGER_DIP_x', 'INDEX_FINGER_DIP_y', 'INDEX_FINGER_DIP_z', 'INDEX_FINGER_TIP_x', 'INDEX_FINGER_TIP_y', 'INDEX_FINGER_TIP_z', 'MIDDLE_FINGER_MCP_x', 'MIDDLE_FINGER_MCP_y', 'MIDDLE_FINGER_MCP_z', 'MIDDLE_FINGER_PIP_x', 'MIDDLE_FINGER_PIP_y', 'MIDDLE_FINGER_PIP_z', 'MIDDLE_FINGER_DIP_x', 'MIDDLE_FINGER_DIP_y', 'MIDDLE_FINGER_DIP_z', 'MIDDLE_FINGER_TIP_x', 'MIDDLE_FINGER_TIP_y', 'MIDDLE_FINGER_TIP_z', 'RING_FINGER_MCP_x', 'RING_FINGER_MCP_y', 'RING_FINGER_MCP_z', 'RING_FINGER_PIP_x', 'RING_FINGER_PIP_y', 'RING_FINGER_PIP_z', 'RING_FINGER_DIP_x', 'RING_FINGER_DIP_y', 'RING_FINGER_DIP_z', 'RING_FINGER_TIP_x', 'RING_FING

C:\Users\btamm\AppData\Local\Temp\ipykernel_24564\2006135473.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gesture_data = pd.concat([gesture_data, pd.DataFrame([nodes], columns=landmark_columns)])
C:\Users\btamm\AppData\Local\Temp\ipykernel_24564\2006135473.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gesture_data = pd.concat([gesture_data, pd.DataFrame([nodes], columns=landmark_columns)])
C:\Users\btamm\AppData\Local\Temp\ipykernel_24564\2006135473.py:39: FutureWarning: The behav